In [32]:
import pandas as pd

top_pairs = pd.read_csv("top_1000_human_mouse_cui_pairs.csv")
top_human_within = pd.read_csv("top_100_within_human_cui_pairs.csv")
top_mouse_within = pd.read_csv("top_100_within_mouse_cui_pairs.csv")
all_top_pairs = pd.read_csv("top_all_cui_similarity_pairs.csv")

human_preds = pd.read_csv("results/top1/human_title_summary_preds.csv")
mouse_preds = pd.read_csv("results/top1/mouse_title_summary_preds.csv")

import json
from pathlib import Path

# metadata human
path = Path("metadata/metadata_human.json")

with open(path, "r", encoding="utf-8") as f:
    data = json.load(f)

rows = []

for gse_id, meta in data.items():
    row = {"gse_id": gse_id}
    row.update(meta)
    rows.append(row)

metadata_human = pd.DataFrame(rows)

# metadata_human.head()

# metadata mouse
path = Path("metadata/metadata_mouse.json")

with open(path, "r", encoding="utf-8") as f:
    data = json.load(f)

rows = []

for gse_id, meta in data.items():
    row = {"gse_id": gse_id}
    row.update(meta)
    rows.append(row)

metadata_mouse = pd.DataFrame(rows)

# metadata_mouse.head()

In [33]:
# exclude pairs with exactly the same CUI set

n_before = len(top_pairs)

exact_same_pairs = top_pairs[
    (top_pairs["jaccard"] == 1.0) &
    (top_pairs["containment"] == 1.0)
]

print(f"Exact same CUI-set pairs found: {len(exact_same_pairs)}")

# optional: inspect them first
# display(exact_same_pairs[[
#     "human_gse", "mouse_gse",
#     "jaccard", "containment", "overlap_count",
#     "human_title", "mouse_title"
# ]].head(30))

top_pairs = top_pairs[
    ~(
        (top_pairs["jaccard"] == 1.0) &
        (top_pairs["containment"] == 1.0)
    )
].reset_index(drop=True)

n_after = len(top_pairs)

print(f"Removed {n_before - n_after} exact same pairs.")
print(f"Remaining pairs: {n_after}")

# top_pairs.to_csv(output_path, index=False)
# print(f"Saved filtered file to: {output_path}")

Exact same CUI-set pairs found: 12
Removed 12 exact same pairs.
Remaining pairs: 988


In [34]:
# remove pairs where human summary and mouse summary are exactly the same

top_pairs["human_summary"] = top_pairs["human_gse"].map(
    metadata_human.set_index("gse_id")["Summary"]
)

top_pairs["mouse_summary"] = top_pairs["mouse_gse"].map(
    metadata_mouse.set_index("gse_id")["Summary"]
)

same_summary_pairs = top_pairs[
    top_pairs["human_summary"] == top_pairs["mouse_summary"]
]

print(f"Pairs with identical summaries: {len(same_summary_pairs)}")

# display(same_summary_pairs[[
#     "human_gse", "mouse_gse",
#     "jaccard", "containment", "overlap_count",
#     "human_title", "mouse_title"
# ]])

top_pairs = top_pairs[
    top_pairs["human_summary"] != top_pairs["mouse_summary"]
].reset_index(drop=True)

# top_pairs.to_csv(output_path, index=False)

Pairs with identical summaries: 30


In [37]:
print(human_preds[human_preds['ID'] == 'GSE83492'])
print(mouse_preds[mouse_preds['ID'] == 'GSE83991'])

            ID       mondo_id      prob  log2(prob/prior) related_words
2486  GSE83492  MONDO_0005061  0.458925          4.678117          lung
            ID       mondo_id      prob  log2(prob/prior) related_words
2953  GSE83991  MONDO_0005061  0.332263          4.212185          lung


In [36]:
top_pairs.head(10)

,human_gse,mouse_gse,jaccard,containment,overlap_count,shared_cuis,human_title,mouse_title,human_summary,mouse_summary
0,GSE95277,GSE95168,0.945055,1.000000,86,"['C0002766', 'C0003516', 'C0003827', 'C0007621...",H3.3K27M cooperates with p53 loss and Pdgfra g...,H3.3K27M cooperates with p53 loss and Pdgfra g...,Gain-of-function mutations in histone 3 (H3) v...,Gain-of-function mutations in histone 3 (H3) v...
1,GSE71777,GSE81149,0.909091,0.967742,90,"['C0007593', 'C0007600', 'C0008546', 'C0008976...","RNASeq of MV4;11 cell treated with DMSO, I-BET...",RNASeq of MLL-AF9 cells transduced with scraml...,Central to the molecular pathogenesis of MLL l...,Central to the molecular pathogenesis of MLL l...
2,GSE79684,GSE79685,0.857143,1.000000,18,"['C0017262', 'C0026691', 'C0035143', 'C0185117...",Loss of CREBBP results in gene expression repr...,Loss of CREBBP results in gene expression repr...,"KD of CREBBP at lymphoma cell line, MD901, res...",KD of Crebbp at mouse cells results in reduced...
3,GSE37061,GSE37018,0.854167,1.000000,41,"['C0002778', 'C0018794', 'C0023621', 'C0027429...",RNA-sequencing analysis of NB4 cells overexpre...,RNA-sequencing analysis of 32Dclone3 cells ove...,To better understand the mechanisms of blockag...,To better understand the mechanisms of blockag...
4,GSE83492,GSE83991,0.772727,0.944444,34,"['C0014597', 'C0024109', 'C0032659', 'C0079366...",Transcriptome analysis of human lung epithelia...,Transcriptome analysis of mouse lung epithelia...,Human lung epithelial subpopulations (alveolar...,Mouse lung epithelial subpopulations (alveolar...
5,GSE50582,GSE49844,0.750000,0.900000,54,"['C0002778', 'C0004793', 'C0006675', 'C0006754...",Transcriptomics analysis of gene expression in...,Transcriptomics analysis of gene expression in...,RNA was isolated from and METTL3ï¼WTAP defici...,RNA was isolated from control and Smg6 deficie...
6,GSE84358,GSE84357,0.724138,0.840000,21,"['C0008972', 'C0017262', 'C0017337', 'C0018017...",RNA-seq of ASXL2 shRNA KD in SKNO-1 cells,RNA-seq of Asxl2 KO LSK cells,The goals of this study are to compare transcr...,The goals of this study are to compare transcr...
7,GSE86910,GSE86865,0.715789,0.944444,68,"['C0002778', 'C0011435', 'C0012929', 'C0014819...",RNA-seq transcriptonal profiling in human prim...,RNA-seq transcriptonal profiling in E13.5 feta...,The developing erythroid cells require highly ...,The developing erythroid cells require highly ...
8,GSE49642,GSE95141,0.714286,1.000000,5,"['C0162327', 'C0162801', 'C0917793', 'C1294197...",Leucegene: AML sequencing (part 1),RNA sequencing of striatum tissue from naive C...,RNA sequencing of human leukemia,RNA sequencing of striatum tissue from naive C...
9,GSE52656,GSE95141,0.714286,1.000000,5,"['C0162327', 'C0162801', 'C0917793', 'C1294197...",Leucegene: AML sequencing (part 2),RNA sequencing of striatum tissue from naive C...,RNA sequencing of human leukemia,RNA sequencing of striatum tissue from naive C...
